# Publication Figure: Comparison of Fed-Batch Feed-Rate Strategies

This notebook builds a **synthetic, publication-focused comparison** of fed-batch feed strategies using the equations in `animakinetica/models/feed_strategies.py` and parameters from the standard config directory.

## Scope and assumptions
- Uses **GFI defaults** from config.
- Uses actual fed-batch volume window from config-derived reactor values:
  - $V_0 = 480\,\mathrm{L}$
  - $V_{\max} = 1600\,\mathrm{L}$
- Constant, exponential, linear: evaluated directly from strategy formulas.
- CSFR: synthetic $X_v(t)$ at constant $\mu=\mu_{\max}$.
- Glucose setpoint: algebraic formula with **constant** $q_{\mathrm{Glc}}$ at $\mu_{\max}$ and $D_{base}=0$.
- Figure is **not** a full coupled ODE simulation; it is intentionally decoupled to isolate strategy behavior.

In [1]:
import sys
from pathlib import Path
from dataclasses import dataclass
import importlib

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure package import from workspace
workspace_root = Path.cwd().parent
src_path = workspace_root / "animakinetica" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Force reload during iterative development so notebook reflects latest code edits
import animakinetica.core.parameters as parameters_module
import animakinetica.models.feed_strategies as feed_strategies_module
import animakinetica.models.stoichiometry as stoichiometry_module

importlib.reload(parameters_module)
importlib.reload(feed_strategies_module)
importlib.reload(stoichiometry_module)

SimulationParameters = parameters_module.SimulationParameters
FeedStrategy = feed_strategies_module.FeedStrategy
create_feed_model = feed_strategies_module.create_feed_model
StoichiometricModel = stoichiometry_module.StoichiometricModel
StoichiometryMode = stoichiometry_module.StoichiometryMode


# -----------------------------------------------------------------------------
# 1) Load standard configuration (GFI defaults) and derive scenario constants
# -----------------------------------------------------------------------------
params = SimulationParameters.load_parameters([
    "base_parameters",
    "operating_modes/fed_batch",
    "cell_properties/cell_GFI",
    "stoichiometry/anab_GFI",
    "stoichiometry/cata_GFI",
])

# -----------------------------------------------------------------------------
# User-configurable synthetic scenario settings
# -----------------------------------------------------------------------------
F_L0_SPECIFIC_LINEAR_EXP = 0.001  # Applied ONLY to linear/exponential strategy runs

V_reactor = float(params.reactor.V_reactor)
V0 = float(params.reactor.V_batch_init)  # Actual fed-batch start volume from config (0.3 * V_max = 480 L)
V_end = float(params.reactor.V_max)      # Actual max working volume from config (1600 L)
F_L_max = float(params.reactor.F_L_max)

Xv0 = float(params.initial_conditions.X_inoc)
# C_Glc0 = float(params.initial_conditions.C_Glc_init)
C_Glc0 = 10
params.set_parameter_value("initial_conditions.C_Glc_init", C_Glc0)

C_Glc_feed = float(params.fed_batch.C_Glc_feed)
C_Glc_set = float(params.fed_batch.C_Glc_setpoint)#15.0  # mmol/L (fixed for publication figure)
# params.set_parameter_value("fed_batch.C_Glc_setpoint", C_Glc_set)
params.set_parameter_value("fed_batch.r", 0.05)
params.set_parameter_value("fed_batch.feed_slope", 0.3)

mu_max = float(params.cell_properties.mu_max)

# Baseline (from parameter files) and manual override values for F_L0
F_L0_specific_from_file = float(params.fed_batch.F_L0_specific)
F_L0_from_file = float(params.fed_batch.F_L0)
F_L0_linear_exp_manual = float(F_L0_SPECIFIC_LINEAR_EXP) * V0

# Glucose-setpoint controller tunings used by the library implementation
C_Glc_setpoint_Kp = float(params.fed_batch.C_Glc_setpoint_Kp) if params.fed_batch.C_Glc_setpoint_Kp is not None else 0.0
C_Glc_setpoint_deadband = float(params.fed_batch.C_Glc_setpoint_deadband) if params.fed_batch.C_Glc_setpoint_deadband is not None else 0.0
C_Glc_setpoint_slew_rate = float(params.fed_batch.C_Glc_setpoint_slew_rate) if params.fed_batch.C_Glc_setpoint_slew_rate is not None else 0.0

# q_Glc at mu_max from STOICHIOMETRIC yield coefficient (mode CONSTANT)
stoich_model = StoichiometricModel(params=params, mode=StoichiometryMode.CONSTANT)
Y_Glc_X_mu_max = float(stoich_model.calculate_yields(mu_max)[0])
cells_per_mmol = float(params.cell_properties.cells_per_mol) / 1000.0
q_Glc_const = (mu_max * Y_Glc_X_mu_max) / cells_per_mmol

D_base_const = 0.0  # user-selected synthetic assumption

print("Scenario loaded")
print(f"V_reactor={V_reactor:.1f} L, V0={V0:.1f} L, Vend={V_end:.1f} L, F_L_max={F_L_max:.2f} L/h")
print(f"Xv0={Xv0:.2e} cells/L, C_Glc0={C_Glc0:.2f} mmol/L")
print(f"mu_max={mu_max:.5f} 1/h, Y_Glc/X(mu_max)={Y_Glc_X_mu_max:.4f} mmol/mmol, q_Glc_const={q_Glc_const:.3e} mmol/cell/h")
print(f"C_Glc_feed={C_Glc_feed:.2f}, C_Glc_setpoint={C_Glc_set:.2f} mmol/L")
print(f"From file -> fed_batch.F_L0_specific={F_L0_specific_from_file:.4f}, fed_batch.F_L0={F_L0_from_file:.3f} L/h")
print(f"Manual (linear/exponential only) -> F_L0_specific={F_L0_SPECIFIC_LINEAR_EXP:.4f}, fed_batch.F_L0={F_L0_linear_exp_manual:.3f} L/h")
print(
    "Glucose-setpoint control tuning: "
    f"Kp={C_Glc_setpoint_Kp:.3f} 1/h, "
    f"deadband={C_Glc_setpoint_deadband:.3f} mmol/L, "
    f"slew_limit={C_Glc_setpoint_slew_rate:.2f} L/h²"
)


# -----------------------------------------------------------------------------
# 2) Synthetic strategy generators using feed_strategies.py equations
# -----------------------------------------------------------------------------
@dataclass
class StrategyTrace:
    name: str
    t: np.ndarray
    F_process: np.ndarray
    F_base: np.ndarray
    F_total: np.ndarray
    V: np.ndarray
    C_Glc: np.ndarray
    Xv: np.ndarray


def integrate_strategy(strategy: FeedStrategy, dt: float = 0.05, t_max: float = 300.0) -> StrategyTrace:
    """
    Synthetic trajectory integration for publication figure.
    Uses feed strategy equations directly from feed_strategies.py via create_feed_model.

    NOTE:
    - CONSTANT, CSFR, GLUCOSE_SETPOINT use parameters from file.
    - EXPONENTIAL and LINEAR use manual F_L0_specific override.
    """
    if strategy in (FeedStrategy.EXPONENTIAL, FeedStrategy.LINEAR):
        strategy_params = params.create_modified_parameter_object({
            "fed_batch.F_L0_specific": F_L0_SPECIFIC_LINEAR_EXP,
        })
    else:
        strategy_params = params

    model = create_feed_model(strategy, strategy_params)

    t_vals = [0.0]
    V_vals = [V0]
    C_vals = [C_Glc0]
    X_vals = [Xv0]
    Fp_vals = [0.0]
    Fb_vals = [D_base_const * V0]
    Ft_vals = [Fp_vals[-1] + Fb_vals[-1]]

    eps = 1e-12

    while V_vals[-1] < V_end - 1e-9 and t_vals[-1] < t_max:
        t = t_vals[-1]
        V = V_vals[-1]
        C = C_vals[-1]
        X = X_vals[-1]

        # Synthetic cell growth at mu_max for all strategies (explicitly requested for CSFR)
        X_next_unclamped = X * np.exp(mu_max * dt)

        # Build minimal y state vector where y[0]=Xv, y[1]=C_Glc and y[5]=V are used by strategies
        y = [0.0] * 11
        y[0] = X
        y[1] = C
        y[5] = V

        F_process = model.calculate_feed_rate(
            t=t,
            y=y,
            q_Glc=q_Glc_const,
            D_base=D_base_const,
            C_Glc_feed=C_Glc_feed,
        )
        F_process = float(max(0.0, min(F_process, F_L_max)))

        # Hit volume endpoint exactly on last step
        if V + F_process * dt > V_end and F_process > eps:
            dt_eff = (V_end - V) / F_process
        else:
            dt_eff = dt

        F_base = D_base_const * V
        F_total = F_process + F_base

        # State updates
        V_next = V + F_process * dt_eff

        # Simple glucose mass balance approximation
        # d(C*V)/dt = F_process*C_feed - F_process*C - q*X*V
        dCdt = (F_process / max(V, eps)) * (C_Glc_feed - C) - q_Glc_const * X
        C_next = max(0.0, C + dCdt * dt_eff)

        X_next = X * np.exp(mu_max * dt_eff)

        t_vals.append(t + dt_eff)
        V_vals.append(V_next)
        C_vals.append(C_next)
        X_vals.append(X_next)
        Fp_vals.append(F_process)
        Fb_vals.append(F_base)
        Ft_vals.append(F_total)

    return StrategyTrace(
        name=str(strategy),
        t=np.asarray(t_vals, dtype=float),
        F_process=np.asarray(Fp_vals, dtype=float),
        F_base=np.asarray(Fb_vals, dtype=float),
        F_total=np.asarray(Ft_vals, dtype=float),
        V=np.asarray(V_vals, dtype=float),
        C_Glc=np.asarray(C_vals, dtype=float),
        Xv=np.asarray(X_vals, dtype=float),
    )


strategies = [
    FeedStrategy.CONSTANT,
    FeedStrategy.EXPONENTIAL,
    FeedStrategy.LINEAR,
    FeedStrategy.CSFR,
    FeedStrategy.GLUCOSE_SETPOINT,
]

traces = {s: integrate_strategy(s) for s in strategies}


# -----------------------------------------------------------------------------
# 3) Metrics table (informative qualities)
# -----------------------------------------------------------------------------
def settling_time_to_band(t: np.ndarray, c: np.ndarray, c_set: float, rel_band: float = 0.05):
    band = rel_band * c_set
    in_band = np.abs(c - c_set) <= band
    for i in range(len(t)):
        if np.all(in_band[i:]):
            return float(t[i])
    return np.nan


rows = []
for s, tr in traces.items():
    cap_mask = np.isclose(tr.F_process, F_L_max, rtol=1e-6, atol=1e-9)
    rows.append({
        "Strategy": s.name,
        "Time to 1600 L (h)": tr.t[-1],
        "Peak Process Feed (L/h)": float(np.max(tr.F_process)),
        "Mean Process Feed (L/h)": float(np.mean(tr.F_process)),
        "Cap-hit Fraction (%)": 100.0 * float(np.mean(cap_mask)),
        "Glucose Min (mmol/L)": float(np.min(tr.C_Glc)),
        "Glucose Final (mmol/L)": float(tr.C_Glc[-1]),
        "Settle to ±5% (h)": settling_time_to_band(tr.t, tr.C_Glc, C_Glc_set, rel_band=0.05),
        "Peak |Glc Error| (mmol/L)": float(np.max(np.abs(tr.C_Glc - C_Glc_set))),
    })

metrics_df = pd.DataFrame(rows).sort_values("Time to 1600 L (h)").reset_index(drop=True)
display(metrics_df.style.format({
    "Time to 1600 L (h)": "{:.1f}",
    "Peak Process Feed (L/h)": "{:.2f}",
    "Mean Process Feed (L/h)": "{:.2f}",
    "Cap-hit Fraction (%)": "{:.1f}",
    "Glucose Min (mmol/L)": "{:.2f}",
    "Glucose Final (mmol/L)": "{:.2f}",
    "Settle to ±5% (h)": "{:.1f}",
    "Peak |Glc Error| (mmol/L)": "{:.2f}",
}))


# -----------------------------------------------------------------------------
# 4) Publication figure (1x3 with horizontal legend below)
# -----------------------------------------------------------------------------
color_map = {
    FeedStrategy.CONSTANT: "#1f77b4",
    FeedStrategy.EXPONENTIAL: "#ff7f0e",
    FeedStrategy.LINEAR: "#2ca02c",
    FeedStrategy.CSFR: "#d62728",
    FeedStrategy.GLUCOSE_SETPOINT: "#9467bd",
}

line_dash_map = {
    FeedStrategy.CONSTANT: "solid",
    FeedStrategy.EXPONENTIAL: "dash",
    FeedStrategy.LINEAR: "dot",
    FeedStrategy.CSFR: "dashdot",
    FeedStrategy.GLUCOSE_SETPOINT: "longdash",
}

line_width_primary = 7
line_width_secondary = 7
axis_line_width = 2.5
tick_width = 2.5

# Axis-range placeholders for this figure (edit these values directly)
# Use None to keep auto-scaling
FEDBATCH_AXIS_RANGES = {
    "A": {"x": None, "y": [0, 100]},  # A) Process Feed Rate e.g., "x": [0, 120], "y": [0, 20]
    "B": {"x": None, "y": [500, 2000]},  # B) Volume
    "C": {"x": None, "y": [0,30]},  # C) Glucose
}

fig = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.10,
)

for s in strategies:
    tr = traces[s]
    label = s.name.replace("_", " ").title()
    color = color_map[s]
    dash = line_dash_map[s]

    fig.add_trace(
        go.Scatter(
            x=tr.t,
            y=tr.F_process,
            mode="lines",
            name=label,
            line=dict(width=line_width_primary, color=color, dash=dash),
            showlegend=True,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=tr.t,
            y=tr.V,
            mode="lines",
            name=label,
            line=dict(width=line_width_secondary, color=color, dash=dash),
            showlegend=False,
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=tr.t,
            y=tr.C_Glc,
            mode="lines",
            name=label,
            line=dict(width=line_width_secondary, color=color, dash=dash),
            showlegend=False,
        ),
        row=1,
        col=3,
    )

# Axis titles for data panels
fig.update_xaxes(title_text="Time (h)", row=1, col=1)
fig.update_xaxes(title_text="Time (h)", row=1, col=2)
fig.update_xaxes(title_text="Time (h)", row=1, col=3)

fig.update_yaxes(title_text="F<sub>Feed</sub> (L h<sup>-1</sup>)", row=1, col=1)
fig.update_yaxes(title_text="V<sub>L</sub> (L)", row=1, col=2)
fig.update_yaxes(title_text="C<sub>Glc</sub> (mmol L<sup>-1</sup>)", row=1, col=3)

fig.add_hline(y=V_end, line_dash="dash", line_color="black", opacity=0.35, row=1, col=2)
fig.add_hline(y=C_Glc_set, line_dash="dash", line_color="#9467bd", opacity=0.30, row=1, col=3)

# Apply optional manual axis ranges
if FEDBATCH_AXIS_RANGES["A"]["x"] is not None:
    fig.update_xaxes(range=FEDBATCH_AXIS_RANGES["A"]["x"], row=1, col=1)
if FEDBATCH_AXIS_RANGES["A"]["y"] is not None:
    fig.update_yaxes(range=FEDBATCH_AXIS_RANGES["A"]["y"], row=1, col=1)

if FEDBATCH_AXIS_RANGES["B"]["x"] is not None:
    fig.update_xaxes(range=FEDBATCH_AXIS_RANGES["B"]["x"], row=1, col=2)
if FEDBATCH_AXIS_RANGES["B"]["y"] is not None:
    fig.update_yaxes(range=FEDBATCH_AXIS_RANGES["B"]["y"], row=1, col=2)

if FEDBATCH_AXIS_RANGES["C"]["x"] is not None:
    fig.update_xaxes(range=FEDBATCH_AXIS_RANGES["C"]["x"], row=1, col=3)
if FEDBATCH_AXIS_RANGES["C"]["y"] is not None:
    fig.update_yaxes(range=FEDBATCH_AXIS_RANGES["C"]["y"], row=1, col=3)

fig.update_xaxes(
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)
fig.update_yaxes(
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)

fig.update_layout(
    template="simple_white",
    height=700,
    width=2000,
    font=dict(size=25, color="black"),
    legend=dict(
        title_text="",
        orientation="h",
        yanchor="top",
        y=-0.35,
        xanchor="center",
        x=0.5,
        traceorder="normal",
        font=dict(size=20),
    ),
    margin=dict(l=100, r=50, t=50, b=220),
)

fig.show()

# Update to transparent background
fig.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",   # whole canvas
    plot_bgcolor="rgba(0,0,0,0)",    # plotting area
    legend=dict(bgcolor="rgba(0,0,0,0)")
)

# -----------------------------------------------------------------------------
# 5) Export for publication/reproducibility
# -----------------------------------------------------------------------------
SAVE_EXPORTS = True

if SAVE_EXPORTS == True:

    out_dir = Path.cwd()
    # csv_path = out_dir / "fedbatch_strategy_comparison_metrics.csv"
    svg_path = out_dir / "fedbatch_strategy_comparison.svg"
    # png_path = out_dir / "fedbatch_strategy_comparison.png"

    # metrics_df.to_csv(csv_path, index=False)
    # print(f"Saved metrics table: {csv_path}")

    try:
        fig.write_image(str(svg_path))
        # fig.write_image(str(png_path), scale=3)
        print(f"Saved figure: {svg_path}")
        # print(f"Saved figure: {png_path}")
    except Exception as exc:
        print("Image export skipped (install/enable kaleido if needed).")
        print(f"Export error: {exc}")

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
  [3/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\cell_GFI.yaml
  [4/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [5/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
Successfully loaded parameters from 5 file(s).
Scenario loaded
V_reactor=2500.0 L, V0=600.0 L, Vend=2000.0 L, F_L_max=400.00 L

,Strategy,Time to 1600 L (h),Peak Process Feed (L/h),Mean Process Feed (L/h),Cap-hit Fraction (%),Glucose Min (mmol/L),Glucose Final (mmol/L),Settle to ±5% (h),Peak |Glc Error| (mmol/L)
0,CONSTANT,77.8,18.00,17.99,0.0,0.84,0.84,nan,7.90
1,GLUCOSE_SETPOINT,81.1,123.57,17.29,0.0,5.00,5.00,47.2,5.00
2,CSFR,85.4,96.11,16.39,0.0,1.11,1.11,nan,5.00
3,LINEAR,94.7,28.99,14.79,0.0,0.00,0.00,nan,5.00
4,EXPONENTIAL,95.4,70.57,14.68,0.0,0.00,0.00,nan,5.00


Saved figure: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Method Figures\fedbatch_strategy_comparison.svg


In [2]:
# -----------------------------------------------------------------------------
# Additional Figure 1A-only: Process Feed Rate panel isolated
# -----------------------------------------------------------------------------
fig_fb_panel_a = go.Figure()

for s in strategies:
    tr = traces[s]
    if s == FeedStrategy.CSFR:
        label = "CSFR"
    else:
        label = s.name.replace("_", " ").title()
    color = color_map[s]
    dash = line_dash_map[s]

    fig_fb_panel_a.add_trace(
        go.Scatter(
            x=tr.t,
            y=tr.F_process,
            mode="lines",
            name=label,
            line=dict(width=8, color=color, dash=dash),
        )
    )

fig_fb_panel_a.update_xaxes(
    title_text="Time (h)",
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)
fig_fb_panel_a.update_yaxes(
    title_text="F<sub>Feed</sub> (L h<sup>-1</sup>)",
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)

if FEDBATCH_AXIS_RANGES["A"]["x"] is not None:
    fig_fb_panel_a.update_xaxes(range=FEDBATCH_AXIS_RANGES["A"]["x"])
if FEDBATCH_AXIS_RANGES["A"]["y"] is not None:
    fig_fb_panel_a.update_yaxes(range=FEDBATCH_AXIS_RANGES["A"]["y"])

fig_fb_panel_a.update_layout(
    template="simple_white",
    height=750,
    width=850,
    font=dict(size=25, color="black"),
    legend=dict(
        title_text="",
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.0)",
        font=dict(size=30),
    ),
    margin=dict(l=100, r=50, t=50, b=100),
)

fig_fb_panel_a.show()

fig_fb_panel_a.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    legend=dict(bgcolor="rgba(0,0,0,0)"),
)

# -----------------------------------------------------------------------------
# 5) Export for publication/reproducibility
# -----------------------------------------------------------------------------
SAVE_EXPORTS = True

if SAVE_EXPORTS == True:

    out_dir = Path.cwd()
    # csv_path = out_dir / "fedbatch_strategy_comparison_metrics.csv"
    svg_path = out_dir / "fedbatch_strategy_comparison_panel_a.svg"
    # png_path = out_dir / "fedbatch_strategy_comparison.png"

    # metrics_df.to_csv(csv_path, index=False)
    # print(f"Saved metrics table: {csv_path}")

    try:
        fig_fb_panel_a.write_image(str(svg_path))
        # fig.write_image(str(png_path), scale=3)
        print(f"Saved figure: {svg_path}")
        # print(f"Saved figure: {png_path}")
    except Exception as exc:
        print("Image export skipped (install/enable kaleido if needed).")
        print(f"Export error: {exc}")

Saved figure: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Method Figures\fedbatch_strategy_comparison_panel_a.svg


Figure 1. Synthetic comparison of fed-batch feed strategies. (A) Process feed-rate profiles generated for constant, exponential, linear, cell-specific-feed-rate, and glucose-setpoint control. (B) Resulting volume trajectories from V0 = 480L to Vmax = 1600L under each feed strategy. (C) Corresponding glucose trajectories. 
All curves are generated from isolated strategy equations and are not coupled to a bioreactor ODE simulation.


# Publication Figure: Comparison of Perfusion Dilution-Rate Strategies

This section builds a **synthetic, publication-focused comparison** of perfusion strategies using the equations in `animakinetica/models/perfusion_strategies.py`.

## Scope and assumptions
- Uses standard GFI-aligned config loading for consistency.
- Uses **yield-derived driver trajectories** for $X_v$, lactate, and ammonia under CONSTANT stoichiometry at $\mu_{\max}$.
- Isolates controller behavior (**not** a coupled ODE simulation).
- Compares all available perfusion strategies:
  - Constant
  - Toxicity-based step (ratchet)
  - Toxicity-based proportional
  - CSPR
  - Glucose setpoint
- For glucose-setpoint perfusion:
  - constant $q_{\mathrm{Glc}}$ at $\mu_{\max}$


In [6]:
from dataclasses import dataclass

from animakinetica.models.perfusion_strategies import PerfusionStrategy, create_perfusion_strategy


# -----------------------------------------------------------------------------
# 1) Load standard configuration and derive scenario constants
# -----------------------------------------------------------------------------
params_perfusion = SimulationParameters.load_parameters([
    "base_parameters",
    "operating_modes/perfusion",
    "cell_properties/cell_GFI",
    "stoichiometry/anab_GFI",
    "stoichiometry/cata_GFI",
])

# Time grid
T_END = 100  # h
DT = 0.25    # h
t_grid = np.arange(0.0, T_END + DT, DT)

# Representative fixed volume for conversion from D (1/h) to flow (L/h)
V_ref = float(params_perfusion.reactor.V_max)
D_max = float(params_perfusion.reactor.D_max)

# Drivers and setpoints
Xv0_pf = float(params_perfusion.initial_conditions.X_inoc)
mu_max_pf = float(params_perfusion.cell_properties.mu_max)

IC100_Lac = float(params_perfusion.kinetics.IC100_Lac)
IC100_Amm = float(params_perfusion.kinetics.IC100_Amm)

C_Glc0_pf = float(params_perfusion.initial_conditions.C_Glc_init)
C_Glc_feed_pf = float(params_perfusion.perfusion.C_Glc_feed)
C_Glc_set_pf = 15.0  # mmol/L (fixed for publication figure)

# Keep chemical dilution fixed in this isolated synthetic comparison
D_base_const_pf = 0.0

# Optional glucose-setpoint perfusion tuning (new controller hooks)
C_Glc_setpoint_Kp_pf = float(params_perfusion.perfusion.C_Glc_setpoint_Kp) if params_perfusion.perfusion.C_Glc_setpoint_Kp is not None else 0.0
C_Glc_setpoint_deadband_pf = float(params_perfusion.perfusion.C_Glc_setpoint_deadband) if params_perfusion.perfusion.C_Glc_setpoint_deadband is not None else 0.0
C_Glc_setpoint_slew_rate_pf = float(params_perfusion.perfusion.C_Glc_setpoint_slew_rate) if params_perfusion.perfusion.C_Glc_setpoint_slew_rate is not None else 0.0

stoich_model_pf = StoichiometricModel(params=params_perfusion, mode=StoichiometryMode.CONSTANT)
Y_Glc_X_mu_max_pf = float(stoich_model_pf.calculate_yields(mu_max_pf)[0])
cells_per_mmol_pf = float(params_perfusion.cell_properties.cells_per_mol) / 1000.0
q_Glc_const_pf = (mu_max_pf * Y_Glc_X_mu_max_pf) / cells_per_mmol_pf

# Yield-derived waste-production rates at CONSTANT stoichiometry
yields_mu_max_pf = stoich_model_pf.calculate_yields(mu_max_pf)
Y_Lac_Glc_pf = float(yields_mu_max_pf[1])
Y_Gln_Glc_pf = float(yields_mu_max_pf[2])
Y_Amm_Gln_pf = float(yields_mu_max_pf[6])
Y_Amm_Glc_pf = Y_Amm_Gln_pf * Y_Gln_Glc_pf
Y_Amm_X_pf = Y_Amm_Glc_pf * Y_Glc_X_mu_max_pf

q_Gln_const_pf = Y_Gln_Glc_pf * q_Glc_const_pf
q_Lac_const_pf = Y_Lac_Glc_pf * q_Glc_const_pf
q_Amm_const_pf = Y_Amm_Glc_pf * q_Glc_const_pf

print("CONSTANT stoichiometry yields at mu_max")
print(f"Y_Lac/Glc={Y_Lac_Glc_pf:.4f}, Y_Gln/Glc={Y_Gln_Glc_pf:.4f}, Y_Amm/Gln={Y_Amm_Gln_pf:.4f}")
print(f"Derived Y_Amm/Glc={Y_Amm_Glc_pf:.4f}, Y_Amm/X={Y_Amm_X_pf:.4f}")
print(f"q_Glc={q_Glc_const_pf:.3e}, q_Gln={q_Gln_const_pf:.3e}, q_Lac={q_Lac_const_pf:.3e}, q_Amm={q_Amm_const_pf:.3e} mmol/cell/h")
print(
    "Perfusion glucose-setpoint tuning: "
    f"Kp={C_Glc_setpoint_Kp_pf:.3f} 1/h, "
    f"deadband={C_Glc_setpoint_deadband_pf:.3f} mmol/L, "
    f"slew_limit={C_Glc_setpoint_slew_rate_pf:.3f} 1/h²"
)


# -----------------------------------------------------------------------------
# 2) Yield-derived driver trajectories
# -----------------------------------------------------------------------------
def build_synthetic_drivers(t: np.ndarray):
    """Create Xv profile and derive Lac/Amm from stoichiometric production rates."""
    # Xv: growth then soft plateau
    Xv_exp = Xv0_pf * np.exp(mu_max_pf * np.minimum(t, 120.0))
    Xv_cap = 5.0e10
    Xv = Xv_cap - (Xv_cap - Xv_exp) * np.exp(-np.maximum(t - 120.0, 0.0) / 36.0)
    Xv = np.minimum(Xv, Xv_cap)

    # Start from configured initial concentrations
    C_Lac = np.zeros_like(t, dtype=float)
    C_Amm = np.zeros_like(t, dtype=float)
    C_Lac[0] = float(params_perfusion.initial_conditions.C_Lac_init)
    C_Amm[0] = float(params_perfusion.initial_conditions.C_Amm_init)

    # dC/dt = q*Xv in this isolated strategy comparison
    for i in range(1, len(t)):
        dt = float(t[i] - t[i - 1])
        r_Lac = q_Lac_const_pf * Xv[i - 1]
        r_Amm = q_Amm_const_pf * Xv[i - 1]

        C_Lac[i] = max(0.0, C_Lac[i - 1] + dt * r_Lac)
        C_Amm[i] = max(0.0, C_Amm[i - 1] + dt * r_Amm)

    return Xv, C_Lac, C_Amm


Xv_drv, C_Lac_drv, C_Amm_drv = build_synthetic_drivers(t_grid)


# -----------------------------------------------------------------------------
# 3) Strategy runner (coupled synthetic D + C_Glc trajectory)
# -----------------------------------------------------------------------------
@dataclass
class PerfusionTrace:
    name: str
    t: np.ndarray
    D: np.ndarray
    F_perf: np.ndarray
    C_Glc: np.ndarray
    level: np.ndarray | None = None


def integrate_perfusion_strategy(strategy: PerfusionStrategy, t: np.ndarray) -> PerfusionTrace:
    """
    Evaluate perfusion strategy on synthetic drivers (no full ODE coupling).

    IMPORTANT: C_Glc is integrated in the same loop used to evaluate D so that
    glucose-setpoint control sees the evolving glucose state (y[1]).
    """
    model = create_perfusion_strategy(strategy, params_perfusion)  # fresh instance (important for ratchet/slew state)

    D_vals = np.zeros_like(t, dtype=float)
    C_vals = np.zeros_like(t, dtype=float)
    level_vals = np.full_like(t, np.nan, dtype=float)

    C_curr = float(C_Glc0_pf)
    C_vals[0] = C_curr

    for i, ti in enumerate(t):
        y = [0.0] * 11
        y[0] = float(Xv_drv[i])
        y[1] = float(C_curr)      # critical for glucose-setpoint control correctness
        y[3] = float(C_Lac_drv[i])
        y[4] = float(C_Amm_drv[i])
        y[5] = float(V_ref)

        D_i = model.calculate_dilution_rate(
            t=float(ti),
            y=y,
            q_Glc=float(q_Glc_const_pf),
            D_base=float(D_base_const_pf),
            C_Glc_feed=float(C_Glc_feed_pf),
        )
        D_vals[i] = float(D_i)

        if strategy == PerfusionStrategy.TOXICITY_BASED_STEP:
            level_vals[i] = float(getattr(model, "_current_level", np.nan))

        # Forward-Euler glucose update for next time sample
        if i < len(t) - 1:
            dt = float(t[i + 1] - t[i])
            dCdt = (
                float(D_i) * (float(C_Glc_feed_pf) - float(C_curr))
                - float(C_curr) * float(D_base_const_pf)
                - float(q_Glc_const_pf) * float(Xv_drv[i])
            )
            C_next = max(0.0, float(C_curr) + dt * dCdt)
            C_vals[i + 1] = C_next
            C_curr = C_next

    F_vals = D_vals * V_ref
    level_arr = level_vals if strategy == PerfusionStrategy.TOXICITY_BASED_STEP else None
    return PerfusionTrace(name=str(strategy), t=t.copy(), D=D_vals, F_perf=F_vals, C_Glc=C_vals, level=level_arr)


perfusion_strategies = [
    PerfusionStrategy.CONSTANT,
    PerfusionStrategy.TOXICITY_BASED_STEP,
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL,
    PerfusionStrategy.CSPR,
    PerfusionStrategy.GLUCOSE_SETPOINT,
]

perfusion_traces = {s: integrate_perfusion_strategy(s, t_grid) for s in perfusion_strategies}


# -----------------------------------------------------------------------------
# 4) Metrics table
# -----------------------------------------------------------------------------
def first_crossing_time(t: np.ndarray, y: np.ndarray, threshold: float):
    idx = np.where(y >= threshold)[0]
    return float(t[idx[0]]) if len(idx) else np.nan


def settling_time_to_band(t: np.ndarray, c: np.ndarray, c_set: float, rel_band: float = 0.05):
    band = rel_band * c_set
    in_band = np.abs(c - c_set) <= band
    for i in range(len(t)):
        if np.all(in_band[i:]):
            return float(t[i])
    return np.nan


rows_pf = []
for s, tr in perfusion_traces.items():
    cap_mask = np.isclose(tr.D, D_max, rtol=1e-6, atol=1e-10)

    row = {
        "Strategy": s.name,
        "Peak D (1/h)": float(np.max(tr.D)),
        "Mean D (1/h)": float(np.mean(tr.D)),
        "Cap-hit Fraction (%)": 100.0 * float(np.mean(cap_mask)),
        "Int Dilution Demand (h^-1·h)": float(np.trapezoid(tr.D, tr.t)),
        "Mean Perfusion Flow (L/h)": float(np.mean(tr.F_perf)),
        "Glucose Min (mmol/L)": float(np.min(tr.C_Glc)),
        "Glucose Final (mmol/L)": float(tr.C_Glc[-1]),
        "Settle to ±5% (h)": settling_time_to_band(tr.t, tr.C_Glc, C_Glc_set_pf, rel_band=0.05),
    }

    if s == PerfusionStrategy.TOXICITY_BASED_STEP and tr.level is not None:
        row["TBS t_level1 (h)"] = first_crossing_time(tr.t, tr.level, 1.0)
        row["TBS t_level2 (h)"] = first_crossing_time(tr.t, tr.level, 2.0)
    else:
        row["TBS t_level1 (h)"] = np.nan
        row["TBS t_level2 (h)"] = np.nan

    if s == PerfusionStrategy.GLUCOSE_SETPOINT:
        # Under nominal assumptions denominator is constant and non-zero.
        den = C_Glc_feed_pf - C_Glc_set_pf
        row["GSP denom near-zero count"] = int(np.sum(np.isclose(den, 0.0, atol=1e-12)))
    else:
        row["GSP denom near-zero count"] = np.nan

    rows_pf.append(row)

metrics_pf_df = pd.DataFrame(rows_pf).sort_values("Int Dilution Demand (h^-1·h)").reset_index(drop=True)
display(metrics_pf_df.style.format({
    "Peak D (1/h)": "{:.4f}",
    "Mean D (1/h)": "{:.4f}",
    "Cap-hit Fraction (%)": "{:.1f}",
    "Int Dilution Demand (h^-1·h)": "{:.2f}",
    "Mean Perfusion Flow (L/h)": "{:.2f}",
    "Glucose Min (mmol/L)": "{:.2f}",
    "Glucose Final (mmol/L)": "{:.2f}",
    "Settle to ±5% (h)": "{:.1f}",
    "TBS t_level1 (h)": "{:.1f}",
    "TBS t_level2 (h)": "{:.1f}",
    "GSP denom near-zero count": "{:.0f}",
}))


# -----------------------------------------------------------------------------
# 5) Publication figure (1x3 with horizontal legend below)
# -----------------------------------------------------------------------------
pf_color_map = {
    PerfusionStrategy.CONSTANT: "#1f77b4",
    PerfusionStrategy.TOXICITY_BASED_STEP: "#ff7f0e",
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL: "#2ca02c",
    PerfusionStrategy.CSPR: "#d62728",
    PerfusionStrategy.GLUCOSE_SETPOINT: "#9467bd",
}

pf_dash_map = {
    PerfusionStrategy.CONSTANT: "solid",
    PerfusionStrategy.TOXICITY_BASED_STEP: "dash",
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL: "dot",
    PerfusionStrategy.CSPR: "dashdot",
    PerfusionStrategy.GLUCOSE_SETPOINT: "longdash",
}

# Convert from cells/L to Million viable cells per mL (MVC/mL)
# 1 MVC/mL = 1e6 cells/mL = 1e9 cells/L
Xv_drv_mvcml = Xv_drv / 1e9

line_width_pf_primary = 7
line_width_pf_secondary = 7
line_width_pf_black = 7
axis_line_width = 2.5
tick_width = 2.5

# Axis-range placeholders for this figure (edit these values directly)
# Use None to keep auto-scaling
PERFUSION_AXIS_RANGES = {
    "A": {"x": None, "y": None},
    "B": {"x": None, "y": None},
    "B_secondary": {"y": None},
    "C": {"x": None, "y": None},
}

fig_pf = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.12,
    specs=[[{}, {"secondary_y": True}, {}]],
)

for s in perfusion_strategies:
    tr = perfusion_traces[s]
    label = s.name.replace("_", " ").title()
    color = pf_color_map[s]
    dash = pf_dash_map[s]

    fig_pf.add_trace(
        go.Scatter(
            x=tr.t,
            y=tr.D,
            mode="lines",
            name=label,
            line=dict(width=line_width_pf_primary, color=color, dash=dash),
            showlegend=True,
        ),
        row=1,
        col=1,
    )
    fig_pf.add_trace(
        go.Scatter(
            x=t_grid,
            y=tr.C_Glc,
            mode="lines",
            name=label,
            line=dict(width=line_width_pf_secondary, color=color, dash=dash),
            showlegend=False,
        ),
        row=1,
        col=3,
    )

# Panel B primary y-axis: toxicity drivers + thresholds (black, thicker, distinct dash)
fig_pf.add_trace(
    go.Scatter(
        x=t_grid,
        y=C_Lac_drv,
        mode="lines",
        name="Lactate",
        line=dict(width=line_width_pf_black, color="black", dash="solid"),
        showlegend=True,
    ),
    row=1,
    col=2,
    secondary_y=False,
)
fig_pf.add_trace(
    go.Scatter(
        x=t_grid,
        y=C_Amm_drv,
        mode="lines",
        name="Ammonia",
        line=dict(width=line_width_pf_black, color="black", dash="dash"),
        showlegend=True,
    ),
    row=1,
    col=2,
    secondary_y=False,
)

# Panel B secondary y-axis: cell density in MVC/mL (black, thicker, distinct dash)
fig_pf.add_trace(
    go.Scatter(
        x=t_grid,
        y=Xv_drv_mvcml,
        mode="lines",
        name="Cell Density",
        line=dict(width=line_width_pf_black, color="black", dash="dot"),
        showlegend=True,
    ),
    row=1,
    col=2,
    secondary_y=True,
)

# Trigger guide lines in black with distinct line types
fig_pf.add_hline(y=0.25 * IC100_Lac, line_dash="dot", line_color="black", opacity=0.40, row=1, col=2)
fig_pf.add_hline(y=0.50 * IC100_Lac, line_dash="dash", line_color="black", opacity=0.45, row=1, col=2)
fig_pf.add_hline(y=0.25 * IC100_Amm, line_dash="longdash", line_color="black", opacity=0.40, row=1, col=2)
fig_pf.add_hline(y=0.50 * IC100_Amm, line_dash="dashdot", line_color="black", opacity=0.45, row=1, col=2)

fig_pf.add_hline(y=D_max, line_dash="dash", line_color="black", opacity=0.35, row=1, col=1)
fig_pf.add_hline(y=C_Glc_set_pf, line_dash="dash", line_color="#9467bd", opacity=0.35, row=1, col=3)

# Axis titles for data panels
fig_pf.update_xaxes(title_text="Time (h)", row=1, col=1)
fig_pf.update_xaxes(title_text="Time (h)", row=1, col=2)
fig_pf.update_xaxes(title_text="Time (h)", row=1, col=3)

fig_pf.update_yaxes(title_text="D (h<sup>-1</sup>)", row=1, col=1)
fig_pf.update_yaxes(title_text="C (mmol L<sup>-1</sup>)", row=1, col=2, secondary_y=False)
fig_pf.update_yaxes(title_text="X<sub>V</sub> (10<sup>6</sup> cells mL<sup>-1</sup>)", row=1, col=2, secondary_y=True)
fig_pf.update_yaxes(title_text="C<sub>Glc</sub> (mmol L<sup>-1</sup>)", row=1, col=3)

# Apply optional manual axis ranges
if PERFUSION_AXIS_RANGES["A"]["x"] is not None:
    fig_pf.update_xaxes(range=PERFUSION_AXIS_RANGES["A"]["x"], row=1, col=1)
if PERFUSION_AXIS_RANGES["A"]["y"] is not None:
    fig_pf.update_yaxes(range=PERFUSION_AXIS_RANGES["A"]["y"], row=1, col=1)

if PERFUSION_AXIS_RANGES["B"]["x"] is not None:
    fig_pf.update_xaxes(range=PERFUSION_AXIS_RANGES["B"]["x"], row=1, col=2)
if PERFUSION_AXIS_RANGES["B"]["y"] is not None:
    fig_pf.update_yaxes(range=PERFUSION_AXIS_RANGES["B"]["y"], row=1, col=2, secondary_y=False)

if PERFUSION_AXIS_RANGES["B_secondary"]["y"] is not None:
    fig_pf.update_yaxes(range=PERFUSION_AXIS_RANGES["B_secondary"]["y"], row=1, col=2, secondary_y=True)

if PERFUSION_AXIS_RANGES["C"]["x"] is not None:
    fig_pf.update_xaxes(range=PERFUSION_AXIS_RANGES["C"]["x"], row=1, col=3)
if PERFUSION_AXIS_RANGES["C"]["y"] is not None:
    fig_pf.update_yaxes(range=PERFUSION_AXIS_RANGES["C"]["y"], row=1, col=3)

fig_pf.update_xaxes(
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)
fig_pf.update_yaxes(
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)

fig_pf.update_layout(
    template="simple_white",
    height=750,
    width=2400,
    font=dict(size=28, color="black"),
    legend=dict(
        title_text="",
        orientation="h",
        yanchor="top",
        y=-0.30,
        xanchor="center",
        x=0.5,
        traceorder="normal",
        font=dict(size=25),
    ),
    margin=dict(l=100, r=50, t=50, b=220),
)

fig_pf.show()


# Update to transparent background
fig_pf.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",   # whole canvas
    plot_bgcolor="rgba(0,0,0,0)",    # plotting area
    legend=dict(bgcolor="rgba(0,0,0,0)")
)


# -----------------------------------------------------------------------------
# 6) Export for publication/reproducibility
# -----------------------------------------------------------------------------
SAVE_EXPORTS_PF = True

if SAVE_EXPORTS_PF == True:
    out_dir_pf = Path.cwd()
    # csv_path_pf = out_dir_pf / "perfusion_strategy_comparison_metrics.csv"
    svg_path_pf = out_dir_pf / "perfusion_strategy_comparison.svg"
    # png_path_pf = out_dir_pf / "perfusion_strategy_comparison.png"

    # metrics_pf_df.to_csv(csv_path_pf, index=False)
    # print(f"Saved metrics table: {csv_path_pf}")

    try:
        fig_pf.write_image(str(svg_path_pf))
        # fig_pf.write_image(str(png_path_pf), scale=3)
        print(f"Saved figure: {svg_path_pf}")
        # print(f"Saved figure: {png_path_pf}")
    except Exception as exc:
        print("Image export skipped (install/enable kaleido if needed).")
        print(f"Export error: {exc}")

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\operating_modes\perfusion.yaml
  [3/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\cell_GFI.yaml
  [4/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [5/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
Successfully loaded parameters from 5 file(s).
CONSTANT stoichiometry yields at mu_max
Y_Lac/Glc=0.5829, Y_Gln/Glc=0.2934, Y_A

,Strategy,Peak D (1/h),Mean D (1/h),Cap-hit Fraction (%),Int Dilution Demand (h^-1·h),Mean Perfusion Flow (L/h),Glucose Min (mmol/L),Glucose Final (mmol/L),Settle to ±5% (h),TBS t_level1 (h),TBS t_level2 (h),GSP denom near-zero count
0,CSPR,0.1693,0.0431,0.0,4.30,86.28,8.87,8.87,nan,nan,nan,nan
1,GLUCOSE_SETPOINT,0.2000,0.0499,8.2,4.98,99.88,11.74,11.74,nan,nan,nan,0
2,CONSTANT,0.0600,0.0600,0.0,6.00,120.00,0.55,0.55,nan,nan,nan,nan
3,TOXICITY_BASED_PROPORTIONAL,0.2000,0.0964,2.5,9.64,192.88,11.45,11.45,nan,nan,nan,nan
4,TOXICITY_BASED_STEP,0.2000,0.0994,20.9,9.93,198.70,11.92,11.92,nan,62.5,79.2,nan


Saved figure: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Method Figures\perfusion_strategy_comparison.svg


In [4]:
# -----------------------------------------------------------------------------
# Additional Figure 2A-only: Dilution Rate panel isolated
# -----------------------------------------------------------------------------
fig_pf_panel_a = go.Figure()

for s in perfusion_strategies:
    tr = perfusion_traces[s]
    if s == PerfusionStrategy.CSPR:
        label = "CSPR"
    else:
        label = s.name.replace("_", " ").title()
    color = pf_color_map[s]
    dash = pf_dash_map[s]

    fig_pf_panel_a.add_trace(
        go.Scatter(
            x=tr.t,
            y=tr.D,
            mode="lines",
            name=label,
            line=dict(width=line_width_pf_primary, color=color, dash=dash),
        )
    )

fig_pf_panel_a.add_hline(y=D_max, line_dash="dash", line_color="black", opacity=0.35)

fig_pf_panel_a.update_xaxes(
    title_text="Time (h)",
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)
fig_pf_panel_a.update_yaxes(
    title_text="D (h<sup>-1</sup>)",
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)

if PERFUSION_AXIS_RANGES["A"]["x"] is not None:
    fig_pf_panel_a.update_xaxes(range=PERFUSION_AXIS_RANGES["A"]["x"])
if PERFUSION_AXIS_RANGES["A"]["y"] is not None:
    fig_pf_panel_a.update_yaxes(range=PERFUSION_AXIS_RANGES["A"]["y"])

fig_pf_panel_a.update_layout(
    template="simple_white",
    height=750,
    width=850,
    font=dict(size=25, color="black"),
    legend=dict(
        title_text="",
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.0)",
        font=dict(size=30),
    ),
    margin=dict(l=100, r=50, t=50, b=100),
)

fig_pf_panel_a.show()

fig_pf_panel_a.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    legend=dict(bgcolor="rgba(0,0,0,0)"),
)

# -----------------------------------------------------------------------------
# 6) Export for publication/reproducibility
# -----------------------------------------------------------------------------
SAVE_EXPORTS_PF = True

if SAVE_EXPORTS_PF == True:
    out_dir_pf = Path.cwd()
    # csv_path_pf = out_dir_pf / "perfusion_strategy_comparison_metrics.csv"
    svg_path_pf = out_dir_pf / "perfusion_strategy_comparison_panel_a.svg"
    # png_path_pf = out_dir_pf / "perfusion_strategy_comparison.png"

    # metrics_pf_df.to_csv(csv_path_pf, index=False)
    # print(f"Saved metrics table: {csv_path_pf}")

    try:
        fig_pf_panel_a.write_image(str(svg_path_pf))
        # fig_pf.write_image(str(png_path_pf), scale=3)
        print(f"Saved figure: {svg_path_pf}")
        # print(f"Saved figure: {png_path_pf}")
    except Exception as exc:
        print("Image export skipped (install/enable kaleido if needed).")
        print(f"Export error: {exc}")

Saved figure: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Method Figures\perfusion_strategy_comparison_panel_a.svg


In [5]:
# Additional rendering of Figure 2 in grayscale (same plot, colors only)
perfusion_gray_map = {
    "Constant": "#000000",
    "Toxicity Based Step": "#a6a6a6",
    "Toxicity Based Proportional": "#2f2f2f",
    "CSPR": "#808080",
    "Glucose Setpoint": "#595959",
    "Lactate": "#000000",
    "Ammonia": "#000000",
    "Cell Density (MVC/mL)": "#000000",
}

fig_pf_bw = go.Figure(fig_pf)

for trace in fig_pf_bw.data:
    if trace.name in perfusion_gray_map:
        trace.line.color = perfusion_gray_map[trace.name]

# Keep identical guide-line structure, recolor only
if fig_pf_bw.layout.shapes:
    for i, shp in enumerate(fig_pf_bw.layout.shapes):
        if i <= 5:
            shp.line.color = "#000000"
        else:
            shp.line.color = "#5a5a5a"

fig_pf_bw.show()

Figure 2. Synthetic comparison of perfusion dilution-rate strategies. (A) Dilution-rate trajectories for constant, toxicity-based step, toxicity-based proportional, cell-specific-perfusion-rate, and glucose-setpoint perfusion control. (B) Toxicity driver trajectories (lactate and ammonia, primary axis), together with viable cell density in MVC/mL (secondary axis) driving the toxicity-based and CSPR strategy, respectively. (C) Glucose trajectories produced by each perfusion strategy. under the same synthetic assumptions.
All curves are generated from isolated strategy equations and are not coupled to a bioreactor ODE simulation.